# Proyecto: Simulación de Inundación

## Sección 1: Modelado del Terreno (Asignado a: Integrante 1)

In [1]:
import numpy as np

class TerrenoSimulacion:
    def __init__(self, filas=10, columnas=10):
        """
        Inicializa la cuadrícula del terreno.
        Convenciones de valores en la matriz:
          0  -> Terreno libre / seguro (Costo estándar)
         1-3 -> Niveles de riesgo por inundación (1 = Bajo, 2 = Medio, 3 = Alto)
         -1  -> Zona bloqueada / intransitable (Muros o inundación crítica)
        """
        self.filas = filas
        self.columnas = columnas
        self.matriz = np.zeros((filas, columnas), dtype=int)
        
        # Coordenadas por defecto para el punto de inicio y el refugio
        self.inicio = (0, 0)
        self.refugio = (filas - 1, columnas - 1)
        
    def configurar_escenario_base(self):
        """Configura un mapa de prueba predeterminado con obstáculos y zonas de riesgo."""
        # Establecemos zonas bloqueadas (-1) simulando desbordamientos impenetrables
        self.matriz[2, 1:6] = -1
        self.matriz[5, 3:9] = -1
        
        # Establecemos diferentes niveles de riesgo de inundación (1, 2, 3)
        self.matriz[1, :] = 1        # Riesgo bajo en toda la fila 1
        self.matriz[3, 0:4] = 2      # Riesgo medio
        self.matriz[4, 4:] = 3       # Riesgo alto
        self.matriz[6:8, 2] = 2      # Zona lateral con riesgo medio
        
        # Asegurar que inicio y refugio estén libres de obstáculos
        self.matriz[self.inicio] = 0
        self.matriz[self.refugio] = 0

    def obtener_costo_celda(self, celda, costo_base=1):
        """
        Calcula el costo de transitar por una celda considerando su nivel de riesgo.
        Si la celda está bloqueada (-1) o fuera de límites, retorna infinito.
        """
        f, c = celda
        
        # Validar límites de la cuadrícula
        if not (0 <= f < self.filas and 0 <= c < self.columnas):
            return float('inf')
            
        nivel_riesgo = self.matriz[f, c]
        
        # Si la zona está completamente bloqueada
        if nivel_riesgo == -1:
            return float('inf')
            
        # Fórmula de costo ponderado: 
        # Costo Base + (Nivel de Riesgo * Factor de penalización por riesgo)
        factor_penalizacion = 3
        costo_total = costo_base + (nivel_riesgo * factor_penalizacion)
        
        return costo_total

    def mostrar_mapa(self):
        """Muestra una representación textual clara de la matriz en consola."""
        print("Leyenda: [ 0 ] Libre | [ 1-3 ] Riesgo de Inundación | [ -1 ] Bloqueado\n")
        print(self.matriz)

# --- PRUEBA LOCAL DEL MÓDULO ---
if __name__ == "__main__":
    # Crear terreno de 10x10
    terreno = TerrenoSimulacion(10, 10)
    terreno.configurar_escenario_base()
    terreno.mostrar_mapa()
    
    # Probar costos de ejemplo
    print("\nPrueba de costos:")
    print(f"Costo celda libre (0,0): {terreno.obtener_costo_celda((0, 0))}")
    print(f"Costo celda con riesgo (1,5): {terreno.obtener_costo_celda((1, 5))}")
    print(f"Costo celda bloqueada (2,2): {terreno.obtener_costo_celda((2, 2))}")

Leyenda: [ 0 ] Libre | [ 1-3 ] Riesgo de Inundación | [ -1 ] Bloqueado

[[ 0  0  0  0  0  0  0  0  0  0]
 [ 1  1  1  1  1  1  1  1  1  1]
 [ 0 -1 -1 -1 -1 -1  0  0  0  0]
 [ 2  2  2  2  0  0  0  0  0  0]
 [ 0  0  0  0  3  3  3  3  3  3]
 [ 0  0  0 -1 -1 -1 -1 -1 -1  0]
 [ 0  0  2  0  0  0  0  0  0  0]
 [ 0  0  2  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0]]

Prueba de costos:
Costo celda libre (0,0): 1
Costo celda con riesgo (1,5): 4
Costo celda bloqueada (2,2): inf


## Sección 2: Dijkstra Adaptado (Asignado a: Integrante 2)

In [3]:
import heapq

class DijkstraEvacuacion:
    def __init__(self, terreno):
        """
        Inicializa el algoritmo de Dijkstra adaptado para la simulación de evacuación.
        Recibe una instancia de TerrenoSimulacion.
        """
        self.terreno = terreno

    def obtener_vecinos(self, celda):
        """
        Retorna las celdas vecinas válidas (arriba, abajo, izquierda, derecha).
        """
        f, c = celda
        movimientos = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        vecinos = []
        
        for df, dc in movimientos:
            nf, nc = f + df, c + dc
            if 0 <= nf < self.terreno.filas and 0 <= nc < self.terreno.columnas:
                if self.terreno.matriz[nf, nc] != -1:
                    vecinos.append((nf, nc))
                    
        return vecinos

    def buscar_ruta(self):
        """
        Ejecuta el algoritmo de Dijkstra para encontrar la ruta óptima
        desde self.terreno.inicio hasta self.terreno.refugio, 
        minimizando el costo acumulado por riesgo de inundación.
        """
        inicio = self.terreno.inicio
        refugio = self.terreno.refugio
        
        pq = [(0, inicio)]
        costos_minimos = {inicio: 0}
        camino_previo = {inicio: None}
        
        while pq:
            costo_actual, celda_actual = heapq.heappop(pq)
            
            if celda_actual == refugio:
                break
                
            if costo_actual > costos_minimos.get(celda_actual, float('inf')):
                continue
                
            for vecino in self.obtener_vecinos(celda_actual):
                costo_movimiento = self.terreno.obtener_costo_celda(vecino)
                
                if costo_movimiento == float('inf'):
                    continue
                    
                nuevo_costo = costo_actual + costo_movimiento
                
                if nuevo_costo < costos_minimos.get(vecino, float('inf')):
                    costos_minimos[vecino] = nuevo_costo
                    camino_previo[vecino] = celda_actual
                    heapq.heappush(pq, (nuevo_costo, vecino))
                    
        ruta = []
        actual = refugio
        
        if actual not in camino_previo and actual != inicio:
            return None, float('inf')
            
        while actual is not None:
            ruta.append(actual)
            actual = camino_previo.get(actual)
            
        ruta.reverse()
        return ruta, costos_minimos.get(refugio, float('inf'))

# --- PRUEBA LOCAL DE LA SECCIÓN 2 ---
if __name__ == "__main__":
    terreno_prueba = TerrenoSimulacion(10, 10)
    terreno_prueba.configurar_escenario_base()
    
    dijkstra = DijkstraEvacuacion(terreno_prueba)
    ruta_optima, costo_total = dijkstra.buscar_ruta()
    
    print("\nResultado de Dijkstra Adaptado:")
    print(f"Costo total de evacuación: {costo_total}")
    print(f"Ruta óptima encontrada: {ruta_optima}")


Resultado de Dijkstra Adaptado:
Costo total de evacuación: 27
Ruta óptima encontrada: [(0, 0), (1, 0), (2, 0), (3, 0), (4, 0), (4, 1), (5, 1), (6, 1), (7, 1), (8, 1), (8, 2), (8, 3), (8, 4), (8, 5), (8, 6), (8, 7), (8, 8), (8, 9), (9, 9)]


## Sección 3: A* con Riesgo (Asignado a: Integrante 3)

## Sección 4: Visualización (Asignado a: Integrante 4)

## Sección 5: Pruebas (Asignado a: Integrante 5)